# Multi-Armed Bandit Practice

- 이름: 이영훈
- 학번: 2632036005
- 과목명: 인공지능특강
- 주제: Multi-Armed Bandit

## 1. 실습 목표

이번 실습에서는 Multi-Armed Bandit 문제를 통해 epsilon-greedy 방식이 어떻게 동작하는지 확인한다.
또한 단일 실행, 반복 평균 실행, non-stationary 환경에서의 차이를 비교한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class Bandit:
    def __init__(self, arms=10):  # arms = 슬롯머신 대수
        self.rates = np.random.rand(arms)  # 슬롯머신 각각의 승률 설정(무작위)

    def play(self, arm):
        rate = self.rates[arm]
        if rate > np.random.rand():
            return 1
        else:
            return 0


class Agent:
    def __init__(self, epsilon, action_size=10):
        self.epsilon = epsilon  # 무작위로 행동할 확률(탐색 확률)
        self.Qs = np.zeros(action_size)
        self.ns = np.zeros(action_size)

    # 슬롯머신의 가치 추정
    def update(self, action, reward):
        self.ns[action] += 1
        self.Qs[action] += (reward - self.Qs[action]) / self.ns[action]

    # 행동 선택(ε-탐욕 정책)
    def get_action(self):
        if np.random.rand() < self.epsilon:
            return np.random.randint(0, len(self.Qs))  # 무작위 행동 선택
        return np.argmax(self.Qs)  # 탐욕 행동 선택

## 2. 실습 1: a_bandit.py

기본적인 stationary multi-armed bandit 환경에서 epsilon-greedy 정책을 사용하여
보상이 높은 arm을 점차 찾아가는 과정을 확인한다.

### 2-1. 실행 결과 예시 1

동일한 코드와 하이퍼파라미터(`steps = 2000`, `epsilon = 0.1`)로 첫 번째 실험을 수행하였다.
이때의 누적 보상 및 평균 승률 그래프는 아래와 같다.

In [ ]:
steps = 2000
epsilon = 0.1

bandit = Bandit()
agent = Agent(epsilon)
total_reward = 0
total_rewards = []  # 보상 합
rates = []          # 승률

for step in range(steps):
    action = agent.get_action()   # 행동 선택
    reward = bandit.play(action)  # 실제로 플레이하고 보상을 받음
    agent.update(action, reward)  # 행동과 보상을 통해 학습
    total_reward += reward

    total_rewards.append(total_reward)       # 현재까지의 보상 합 저장
    rates.append(total_reward / (step + 1))  # 현재까지의 승률 저장

print("누적 보상:", total_reward)
print("최종 승률:", rates[-1])

# [그림 1] 스텝별 보상 총합
plt.ylabel('Total reward')
plt.xlabel('Steps')
plt.plot(total_rewards)
plt.show()

# [그림 2] 스텝별 승률
plt.ylabel('Rates')
plt.xlabel('Steps')
plt.plot(rates)
plt.show()

### 2-2. 실행 결과 예시 2

동일한 코드와 하이퍼파라미터(`steps = 2000`, `epsilon = 0.1`)로 두 번째 실험을 수행하였다.
이때의 누적 보상 및 평균 승률 그래프는 아래와 같다.

In [ ]:
steps = 2000
epsilon = 0.1

bandit = Bandit()
agent = Agent(epsilon)
total_reward = 0
total_rewards = []  # 보상 합
rates = []          # 승률

for step in range(steps):
    action = agent.get_action()
    reward = bandit.play(action)
    agent.update(action, reward)
    total_reward += reward

    total_rewards.append(total_reward)
    rates.append(total_reward / (step + 1))

print("누적 보상:", total_reward)
print("최종 승률:", rates[-1])

# [그림 1] 스텝별 보상 총합
plt.ylabel('Total reward')
plt.xlabel('Steps')
plt.plot(total_rewards)
plt.show()

# [그림 2] 스텝별 승률
plt.ylabel('Rates')
plt.xlabel('Steps')
plt.plot(rates)
plt.show()

두 실험은 동일한 코드와 동일한 설정으로 수행되었음에도 불구하고,
누적 보상 곡선과 평균 승률, 최종 승률 값이 서로 다르게 나타났다.
이는 각 실행마다 슬롯머신의 실제 보상 확률이 무작위로 생성되며,
epsilon-greedy 정책 또한 탐색 과정에서 확률적 선택을 수행하기 때문이다.

즉, Multi-Armed Bandit 문제는 본질적으로 확률적 환경에 기반하므로,
동일한 코드로도 실행마다 다른 결과가 나타날 수 있다.
따라서 성능을 보다 신뢰성 있게 해석하기 위해서는 단일 실행 결과뿐 아니라
반복 실행을 통한 평균 성능도 함께 확인할 필요가 있다.

In [ ]:
# [그림 3] 스텝별 승률의 10번의 결과를 한꺼번에 그리기
steps = 2000
epsilon = 0.1

for _ in range(10):
    bandit = Bandit()
    agent = Agent(epsilon)
    total_reward = 0
    rates = []

    for step in range(steps):
        action = agent.get_action()
        reward = bandit.play(action)
        agent.update(action, reward)
        total_reward += reward
        rates.append(total_reward / (step + 1))

    plt.plot(rates)

plt.ylabel('Rates')
plt.xlabel('Steps')
plt.show()

## 3. 실습 2: b_bandit_avg.py

단일 실행 결과는 무작위성의 영향으로 그래프 형태와 최종 승률이 달라질 수 있다.
따라서 동일한 stationary multi-armed bandit 실험을 여러 번 반복 수행하고,
각 step에서의 평균 승률을 계산하여 epsilon-greedy 정책의 평균적인 성능 경향을 확인한다.

In [ ]:
runs = 200
steps = 2000
epsilon = 0.1
all_rates = np.zeros((runs, steps))

for run in range(runs):
    bandit = Bandit()
    agent = Agent(epsilon)
    total_reward = 0
    rates = []

    for step in range(steps):
        action = agent.get_action()
        reward = bandit.play(action)
        agent.update(action, reward)
        total_reward += reward
        rates.append(total_reward / (step + 1))

    all_rates[run] = rates

avg_rates = np.average(all_rates, axis=0)

plt.ylabel('Rates')
plt.xlabel('Steps')
plt.plot(avg_rates)
plt.show()

동일한 stationary multi-armed bandit 실험을 여러 번 반복한 결과, 단일 실행에서 나타났던 그래프의 변동성이 평균화되면서 전체적인 성능 경향을 보다 안정적으로 확인할 수 있었다. 즉, 개별 실행에서는 누적 보상 곡선과 평균 승률, 최종 승률 값이 서로 다르게 나타날 수 있지만, 반복 실험을 통해 평균을 계산하면 epsilon-greedy 정책의 전반적인 학습 특성을 더 명확하게 파악할 수 있다.

이는 각 실행마다 슬롯머신의 실제 보상 확률이 무작위로 생성되고, agent의 행동 선택 또한 확률적 요소를 포함하기 때문이다. 따라서 Multi-Armed Bandit 문제의 성능을 보다 신뢰성 있게 해석하기 위해서는 단일 실행 결과만 확인하는 것보다, 여러 번의 반복 실험을 수행한 뒤 평균 성능을 함께 분석하는 것이 더 적절하다고 볼 수 있다.

In [ ]:
runs = 200
steps = 2000

for epsilon in [0.1, 0.3, 0.01]:
    all_rates = np.zeros((runs, steps))

    for run in range(runs):
        bandit = Bandit()
        agent = Agent(epsilon)
        total_reward = 0
        rates = []

        for step in range(steps):
            action = agent.get_action()
            reward = bandit.play(action)
            agent.update(action, reward)
            total_reward += reward
            rates.append(total_reward / (step + 1))

        all_rates[run] = rates

    avg_rates = np.average(all_rates, axis=0)
    plt.plot(avg_rates, label=str(epsilon))

plt.ylabel('Rates')
plt.xlabel('Steps')
plt.legend()
plt.show()

## 4. 실습 3: c_non_stationary_bandit_avg.py

non-stationary multi-armed bandit 환경에서는 각 슬롯머신의 보상 확률이 시간에 따라 변한다.
이러한 환경에서 기존의 평균 기반 가치 추정 방식이 어떤 한계를 가지는지 확인하고,
환경 변화에 대한 agent의 적응 특성을 살펴본다.

In [ ]:
class NonStationaryBandit:
    def __init__(self, arms=10):
        self.arms = arms
        self.rates = np.random.rand(arms)

    def play(self, arm):
        rate = self.rates[arm]
        self.rates += 0.1 * np.random.randn(self.arms)  # 노이즈 추가
        if rate > np.random.rand():
            return 1
        else:
            return 0


class AlphaAgent:
    def __init__(self, epsilon, alpha, action_size=10):
        self.epsilon = epsilon  # 무작위로 행동할 확률(탐색 확률)
        self.Qs = np.zeros(action_size)
        self.alpha = alpha  # 고정값 α

    def update(self, action, reward):
        self.Qs[action] += (reward - self.Qs[action]) * self.alpha

    def get_action(self):
        if np.random.rand() < self.epsilon:
            return np.random.randint(0, len(self.Qs))
        return np.argmax(self.Qs)


runs = 200
steps = 2000
epsilon = 0.1
alpha = 0.8
agent_types = ['sample average', 'alpha const update']
results = {}

for agent_type in agent_types:
    all_rates = np.zeros((runs, steps))

    for run in range(runs):
        if agent_type == 'sample average':
            agent = Agent(epsilon)
        else:
            agent = AlphaAgent(epsilon, alpha)

        bandit = NonStationaryBandit()
        total_reward = 0
        rates = []

        for step in range(steps):
            action = agent.get_action()
            reward = bandit.play(action)
            agent.update(action, reward)
            total_reward += reward
            rates.append(total_reward / (step + 1))

        all_rates[run] = rates

    avg_rates = np.average(all_rates, axis=0)
    results[agent_type] = avg_rates

# [그림] 표본 평균과 고정값 α에 의한 갱신 비교
plt.figure()
plt.ylabel('Average Rates')
plt.xlabel('Steps')
for key, avg_rates in results.items():
    plt.plot(avg_rates, label=key)
plt.legend()
plt.show()

non-stationary multi-armed bandit 환경에서 두 방식을 비교한 결과, sample average 방식과 alpha const update 방식은 평균 승률 곡선과 최종 성능에서 차이를 보였다. 이는 non-stationary 환경에서는 각 슬롯머신의 보상 확률이 시간에 따라 계속 변하기 때문에, 과거 보상을 동일한 비중으로 반영하는 방식과 최근 보상에 더 큰 가중치를 두는 방식 사이에 적응 속도의 차이가 발생하기 때문이다.

즉, non-stationary multi-armed bandit 문제에서는 단순히 과거 평균을 누적하는 방식보다, 최근 환경 변화를 더 빠르게 반영할 수 있는 업데이트 방식이 더 유리할 수 있다. 따라서 환경이 고정되어 있지 않은 경우에는 sample average 방식의 한계를 고려해야 하며, 성능을 보다 적절하게 해석하기 위해서는 환경 변화에 대한 적응 특성까지 함께 살펴볼 필요가 있다.